In [0]:
import dlt

In [0]:
#create a streaming table for orders

@dlt.table(
    table_properties = {
        "quality":"bronze"
    }, comment ="order bronze table"
    
)
def orders_bronze():
    df = spark.readStream.table("dev.bronze.orders_raw") # stream data
    return df

In [0]:
#create a materialized view for customer

@dlt.table(
    table_properties = {
        "quality":"bronze"
    }, comment ="customer bronze table",
    name ="customer_bronze"
    
)
def cust_bronze():
    df = spark.read.table("dev.bronze.customers_raw")  # this the batch data
    return df

In [0]:
#create a view join orders with customers

@dlt.view(
    comment ="joined View",
   
    
)
def joined_vw():
    df_c = spark.read.table("LIVE.customer_bronze")  # this the batch data
    df_o = spark.read.table("LIVE.orders_bronze") 
    df_join = df_o.join(df_c, how = "left_outer", on = df_c.c_custkey == df_o.o_cust_key)
    return df_join

In [0]:
from pyspark.sql.functions import current_timestamp

@dlt.table(
    table_properties = {
        "quality":"silver"
    }, comment ="joined table",
    name ="joined_silver"
    
)
def joined_silver():
    df = spark.read.table("LIVE.joined_vw").withColumn("__insert_date", current_timestamp())  # this the batch data
    return df

In [0]:
from pyspark.sql.functions import current_timestamp, count

@dlt.table(
    table_properties = {
        "quality":"gold"
    }, comment ="joined table"  
    
)
def orders_agg_gold():
    df = spark.read.table("LIVE.joined_silver")
    df_final = df.groupBy("c_mktsegment").agg(count("c_orderkey").alias("sum_orders")).withColumn("__insert_date", current_timestamp())  # this the batch data
    return df_final